# Notebook 05: Named Entity Retention — Content Erasure Analysis
**Ship of Theseus — Computational Forensics Project**

Do LLM paraphrasers preserve the **concrete content** (named entities: people, places,
organizations, dates) across iterations, or do they silently **drop** and **hallucinate**
entities?

This notebook complements the stylometric analysis (nb03) by measuring
**content-level** erasure rather than **style-level** drift, completing the
multi-modal audit: skeleton (syntax) vs. skin (vocabulary) vs. cargo (entities).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path().resolve().parents[0]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import DATA_PROCESSED, DATASETS, STAGES, FAMILIES

FIGURES = ROOT / "figures" / "ner"
EXP_DATA = ROOT / "experiments" / "ner"
FIGURES.mkdir(parents=True, exist_ok=True)
EXP_DATA.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")

## Load Chain Data
Load T0 → T3 chains from all 7 datasets (500 samples each).

In [ ]:
SAMPLE_N = 500
SEED = 42

chains = {}
for name in DATASETS.keys():
    path = DATA_PROCESSED / f"{name}_chains.csv"
    df = pd.read_csv(path)
    chains[name] = df.sample(min(SAMPLE_N, len(df)), random_state=SEED)
    print(f"Loaded '{name}': {chains[name].shape}")

combined = pd.concat(chains.values(), ignore_index=True)
combined['dataset'] = np.concatenate([
    [name] * len(chains[name]) for name in chains
])
print(f"\nCombined: {combined.shape}")
print(f"Families: {combined['family'].value_counts().to_dict()}")

---
## Section 1: NER Extraction

Extract named entity sets from each stage (T0–T3) using spaCy.
We disable all pipeline components except NER for speed.

In [ ]:
from src.features.ner import extract_ner_for_chains, compute_retention_for_chains

print(f"Extracting NER for {len(combined)} chains across 4 stages...")
print("(This may take a few minutes)\n")

combined = extract_ner_for_chains(combined, stages=STAGES)

# Quick sanity check
sample_row = combined.iloc[0]
print(f"\nSanity check — row 0:")
print(f"  T0 entities ({len(sample_row['ner_T0'])}): {list(sample_row['ner_T0'])[:5]}")
print(f"  T1 entities ({len(sample_row['ner_T1'])}): {list(sample_row['ner_T1'])[:5]}")

## Section 2: Entity Retention Metrics

For each chain, compute **Jaccard**, **Recall** (entity preservation),
and **Precision** (1 − hallucination rate) comparing each stage to T0.

- **Recall < 1** → original entities were **dropped**
- **Precision < 1** → new entities were **hallucinated**

In [ ]:
combined = compute_retention_for_chains(combined, stages=["T1", "T2", "T3"])

# Summary table: mean retention by stage
summary_rows = []
for stage in ["T1", "T2", "T3"]:
    summary_rows.append({
        "Stage": stage,
        "Jaccard": combined[f"jaccard_{stage}"].mean(),
        "Recall": combined[f"recall_{stage}"].mean(),
        "Precision": combined[f"precision_{stage}"].mean(),
        "Mean T0 entities": combined[f"n_ref_{stage}"].mean(),
        "Mean Tn entities": combined[f"n_hyp_{stage}"].mean(),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n=== Entity Retention Summary (all datasets, all families) ===")
print(summary_df.to_string(index=False, float_format="{:.4f}".format))
summary_df.to_csv(EXP_DATA / "ner_retention_summary.csv", index=False)

In [ ]:
# ── NER Decay: Single averaged graph, metrics as lines (T0 → T3) ────────────
STAGES      = ['T0', 'T1', 'T2', 'T3']
PLOT_STAGES = ['T1', 'T2', 'T3']

METRIC_STYLE = {
    'Jaccard':   {'color': '#1565C0', 'ls': '-',  'marker': 'o'},
    'Recall':    {'color': '#2E7D32', 'ls': '--', 'marker': 's'},
    'Precision': {'color': '#C62828', 'ls': ':',  'marker': '^'},
}

# Average across all datasets
overall = domain_df.groupby('Stage')[['Jaccard', 'Recall', 'Precision']].mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_facecolor('#FFFFFF')

for metric, style in METRIC_STYLE.items():
    vals = [1.0] + [overall.loc[s, metric] for s in PLOT_STAGES]
    ax.plot(STAGES, vals,
            color=style['color'], linestyle=style['ls'],
            marker=style['marker'], linewidth=2.2,
            markersize=8, label=metric)
    ax.annotate(f"{vals[-1]:.2f}",
                xy=(3, vals[-1]), xytext=(0, 8),
                textcoords='offset points',
                fontsize=9, color=style['color'],
                ha='center', fontweight='bold')

ax.set_xlabel('Paraphrase Stage', fontsize=11)
ax.set_ylabel('Score (T0 = 1.0)', fontsize=11)
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color='grey', linewidth=0.7, linestyle='--', alpha=0.4)
ax.grid(alpha=0.25)
ax.set_xticks(range(4))
ax.set_xticklabels(STAGES)
ax.legend(fontsize=10)

ax.set_title('NER Entity Retention Decay  (T0 → T3)\n'
             'Averaged across all datasets',
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES / 'fig1.3_ner_decay_averaged.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_ner_decay_averaged.png')

---
## Figure 1: Entity Retention Decay Curves by Family

How does entity overlap (Jaccard) with T0 degrade across iterations,
stratified by paraphraser family?

In [ ]:
family_colors = {
    'chatgpt': '#E53935', 'dipper': '#1E88E5',
    'pegasus': '#43A047', 'palm': '#FDD835',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(axes,
    ['jaccard', 'recall', 'precision'],
    ['Jaccard Similarity', 'Recall (Entity Preservation)', 'Precision (1 − Hallucination)']):

    for family, color in family_colors.items():
        fam_df = combined[combined['family'] == family]
        means = [1.0]  # T0 vs T0 = 1.0
        for stage in ['T1', 'T2', 'T3']:
            means.append(fam_df[f"{metric}_{stage}"].mean())
        ax.plot(STAGES, means, 'o-', color=color, label=family.capitalize(),
                linewidth=2, markersize=6)

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Paraphrase Stage')
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Score (vs. T0)')
plt.suptitle('Entity Retention Across Paraphrase Iterations', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "fig1_ner_decay_by_family.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Section 3: Per-Family Comparison

Aggregate retention metrics by paraphraser family.
Which family drops the most entities? Which hallucinates the most?

In [ ]:
family_rows = []
for family in family_colors.keys():
    fam_df = combined[combined['family'] == family]
    for stage in ['T1', 'T2', 'T3']:
        family_rows.append({
            'Family': family.capitalize(),
            'Stage': stage,
            'Jaccard': fam_df[f'jaccard_{stage}'].mean(),
            'Recall': fam_df[f'recall_{stage}'].mean(),
            'Precision': fam_df[f'precision_{stage}'].mean(),
            'N': len(fam_df),
        })

family_df = pd.DataFrame(family_rows)
print("\n=== Entity Retention by Paraphraser Family ===")
print(family_df.to_string(index=False, float_format="{:.4f}".format))
family_df.to_csv(EXP_DATA / "ner_retention_by_family.csv", index=False)

### Figure 2: Dropping vs. Hallucination by Family (T3)

Scatter plot: Recall (x) vs. Precision (y) at T3.
- **Low recall** = entities dropped (content loss)
- **Low precision** = entities hallucinated (content fabrication)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for family, color in family_colors.items():
    fam_df = combined[combined['family'] == family]
    r = fam_df['recall_T3'].mean()
    p = fam_df['precision_T3'].mean()
    ax.scatter(r, p, color=color, s=200, zorder=5, edgecolors='black', linewidths=1.5)
    ax.annotate(family.capitalize(), (r, p), textcoords='offset points',
                xytext=(10, 10), fontsize=12, fontweight='bold')

# Also plot per-paraphraser variants as smaller dots
for para in combined['paraphraser'].unique():
    para_df = combined[combined['paraphraser'] == para]
    if len(para_df) < 10:
        continue
    r = para_df['recall_T3'].mean()
    p = para_df['precision_T3'].mean()
    fam = para_df['family'].iloc[0]
    ax.scatter(r, p, color=family_colors.get(fam, 'gray'), s=60, alpha=0.6,
              edgecolors='black', linewidths=0.5)
    ax.annotate(para, (r, p), textcoords='offset points',
                xytext=(8, -8), fontsize=8, alpha=0.7)

ax.set_xlabel('Recall (Entity Preservation) →', fontsize=12)
ax.set_ylabel('Precision (1 − Hallucination) →', fontsize=12)
ax.set_title('Entity Dropping vs. Hallucination at T3', fontsize=14, fontweight='bold')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / "fig2_drop_vs_hallucinate.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── NER Decay: Per-Family subplots, metrics as lines (T0 → T3) ──────────────
STAGES      = ['T0', 'T1', 'T2', 'T3']
PLOT_STAGES = ['T1', 'T2', 'T3']

METRIC_STYLE = {
    'Jaccard':   {'color': '#1565C0', 'ls': '-',  'marker': 'o'},
    'Recall':    {'color': '#2E7D32', 'ls': '--', 'marker': 's'},
    'Precision': {'color': '#C62828', 'ls': ':',  'marker': '^'},
}

families_list = list(family_colors.keys())
ncols = 2
nrows = math.ceil(len(families_list) / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), sharey=True)
axes = axes.flatten()

for ax, family in zip(axes, families_list):
    ax.set_facecolor('#FFFFFF')
    fam_rows = family_df[family_df['Family'] == family.capitalize()].set_index('Stage')

    for metric, style in METRIC_STYLE.items():
        vals = [1.0] + [fam_rows.loc[s, metric] for s in PLOT_STAGES]
        ax.plot(STAGES, vals,
                color=style['color'], linestyle=style['ls'],
                marker=style['marker'], linewidth=2.2,
                markersize=7, label=metric)

        ax.annotate(f"{vals[-1]:.2f}",
                    xy=(3, vals[-1]),
                    xytext=(0, 6), textcoords='offset points',
                    fontsize=7.5, color=style['color'],
                    ha='center', fontweight='bold')

    ax.set_title(family.capitalize(), fontweight='bold', fontsize=11)
    ax.set_xlabel('Paraphrase Stage', fontsize=9)
    ax.set_ylabel('Score (T0 = 1.0)', fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_xticks(range(4))
    ax.set_xticklabels(STAGES)
    ax.axhline(1.0, color='grey', linewidth=0.7, linestyle='--', alpha=0.4)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5)

for ax in axes[len(families_list):]:
    ax.set_visible(False)

fig.suptitle('NER Entity Retention Decay  (T0 → T3) — Per Paraphraser Family\n'
             'Jaccard = overlap  ·  Recall = preservation  ·  Precision = 1 − hallucination',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES / 'fig2.2_ner_decay_per_family.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_ner_decay_per_family.png')

---
## Section 4: Per-Domain Analysis

Which text domains are most vulnerable to entity loss?
Scientific text with specialized entities (sci_gen) vs. colloquial text (yelp).

In [ ]:
domain_rows = []
for ds in DATASETS.keys():
    ds_df = combined[combined['dataset'] == ds]
    for stage in ['T1', 'T2', 'T3']:
        domain_rows.append({
            'Dataset': ds,
            'Stage': stage,
            'Jaccard': ds_df[f'jaccard_{stage}'].mean(),
            'Recall': ds_df[f'recall_{stage}'].mean(),
            'Precision': ds_df[f'precision_{stage}'].mean(),
            'Mean T0 entities': ds_df['n_ref_T1'].mean(),
        })

domain_df = pd.DataFrame(domain_rows)
print("\n=== Entity Retention by Domain ===")
print(domain_df.to_string(index=False, float_format="{:.4f}".format))
domain_df.to_csv(EXP_DATA / "ner_retention_by_domain.csv", index=False)

### Figure 3: Per-Domain Entity Retention Heatmap (T3)

In [ ]:
# Pivot for heatmap: rows = datasets, columns = metrics at T3
t3_domain = domain_df[domain_df['Stage'] == 'T3'][['Dataset', 'Jaccard', 'Recall', 'Precision']]
t3_domain = t3_domain.set_index('Dataset').sort_values('Jaccard', ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(t3_domain, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Score'})
ax.set_title('Entity Retention at T3 by Domain', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig(FIGURES / "fig3_domain_retention_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5: Paraphraser Intensity Comparison

Does increasing paraphraser "strength" proportionally increase entity erasure?
- **Pegasus(slight)** vs. **Pegasus(full)**
- **Dipper(low)** vs. **Dipper(high)**

In [ ]:
intensity_pairs = [
    ('pegasus(slight)', 'pegasus(full)', 'Pegasus'),
    ('dipper(low)', 'dipper(high)', 'Dipper'),
]

intensity_rows = []
for low, high, label in intensity_pairs:
    for variant in [low, high]:
        v_df = combined[combined['paraphraser'] == variant]
        if len(v_df) == 0:
            continue
        for stage in ['T1', 'T2', 'T3']:
            intensity_rows.append({
                'Family': label,
                'Variant': variant,
                'Stage': stage,
                'Jaccard': v_df[f'jaccard_{stage}'].mean(),
                'Recall': v_df[f'recall_{stage}'].mean(),
                'Precision': v_df[f'precision_{stage}'].mean(),
                'N': len(v_df),
            })

intensity_df = pd.DataFrame(intensity_rows)
print("\n=== Intensity Comparison ===")
print(intensity_df.to_string(index=False, float_format="{:.4f}".format))
intensity_df.to_csv(EXP_DATA / "ner_intensity_comparison.csv", index=False)

### Figure 4: Intensity — Low vs. High Paraphraser Settings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

variant_colors = {
    'pegasus(slight)': '#81C784', 'pegasus(full)': '#2E7D32',
    'dipper(low)': '#64B5F6', 'dipper(high)': '#1565C0',
}
variant_styles = {
    'pegasus(slight)': '--', 'pegasus(full)': '-',
    'dipper(low)': '--', 'dipper(high)': '-',
}

for ax, (low, high, label) in zip(axes, intensity_pairs):
    for variant in [low, high]:
        v_data = intensity_df[intensity_df['Variant'] == variant]
        if len(v_data) == 0:
            continue
        means = [1.0] + v_data['Jaccard'].tolist()
        ax.plot(STAGES, means, 'o' + variant_styles[variant],
                color=variant_colors[variant], label=variant,
                linewidth=2, markersize=6)

        # Also plot recall as dashed with triangles
        recall_means = [1.0] + v_data['Recall'].tolist()
        ax.plot(STAGES, recall_means, 's' + variant_styles[variant],
                color=variant_colors[variant], alpha=0.5,
                linewidth=1.5, markersize=5, label=f"{variant} (recall)")

    ax.set_title(f'{label} Intensity Comparison', fontsize=12, fontweight='bold')
    ax.set_xlabel('Paraphrase Stage')
    ax.set_ylabel('Score (vs. T0)')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Paraphraser Intensity on Entity Retention',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "fig4_intensity_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6: Structural vs. Content Decay — Multi-Modal Audit

Side-by-side comparison of **skeleton** (POS cosine similarity) vs.
**cargo** (NER Jaccard) vs. **semantic hull** (BERTScore/SBERT).

This is the key "multi-modal audit" figure: did the syntactic skeleton
remain more stable than the entity cargo?

In [ ]:
# Compute POS cosine similarity for the multi-modal audit
from src.features.stylometry import TRACKED_POS
from collections import Counter
import spacy

nlp_pos = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])

# Subsample for speed
audit_sample = combined.sample(min(1000, len(combined)), random_state=SEED).copy()

print("Computing POS vectors for multi-modal audit...")
for stage in STAGES:
    print(f"  {stage}...")
    texts = [str(t)[:5000] if isinstance(t, str) and t.strip() else "" for t in audit_sample[stage]]
    vectors = []
    for doc in nlp_pos.pipe(texts, batch_size=500):
        counts = Counter(t.pos_ for t in doc)
        total = sum(counts.values())
        if total == 0:
            vectors.append(np.zeros(len(TRACKED_POS)))
        else:
            vectors.append(np.array([counts.get(tag, 0) / total for tag in TRACKED_POS]))
    audit_sample[f'pos_{stage}'] = vectors

# POS cosine similarity (T0 vs Tn) — vectorized
for stage in ['T1', 'T2', 'T3']:
    v0 = np.vstack(audit_sample['pos_T0'].values)
    vn = np.vstack(audit_sample[f'pos_{stage}'].values)
    dot = (v0 * vn).sum(axis=1)
    norm0 = np.linalg.norm(v0, axis=1)
    normn = np.linalg.norm(vn, axis=1)
    denom = norm0 * normn
    sims = np.where(denom > 0, dot / denom, np.nan)
    audit_sample[f'pos_cos_{stage}'] = sims

print("Done.")

In [ ]:
# Load SBERT/BERTScore means from baseline experiments if available
sbert_path = ROOT / "experiments" / "baseline_similarity" / "sbert_cosine_results.csv"

# Build the multi-modal comparison
modal_data = {'Stage': STAGES}

# NER Jaccard (content/cargo)
modal_data['NER Jaccard\n(Content)'] = [1.0] + [
    combined[f'jaccard_{s}'].mean() for s in ['T1', 'T2', 'T3']
]

# POS Cosine (structural/skeleton)
modal_data['POS Cosine\n(Structure)'] = [1.0] + [
    audit_sample[f'pos_cos_{s}'].mean() for s in ['T1', 'T2', 'T3']
]

# Semantic similarity from prior work (hardcoded from update2.tex Table 2)
modal_data['BERTScore F1\n(Semantics)'] = [1.0, 0.91, 0.86, 0.81]
modal_data['SBERT Cosine\n(Semantics)'] = [1.0, 0.79, 0.75, 0.73]
modal_data['BLEU-2\n(Lexical)'] = [1.0, 0.54, 0.38, 0.28]

modal_df = pd.DataFrame(modal_data)
print("\n=== Multi-Modal Decay Comparison ===")
print(modal_df.to_string(index=False, float_format="{:.4f}".format))

### Figure 5: Multi-Modal Audit — Skeleton vs. Skin vs. Cargo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

plot_series = {
    'POS Cosine (Structure)':   {'vals': modal_data['POS Cosine\n(Structure)'],  'color': '#1565C0', 'ls': '-',  'marker': 'o'},
    'NER Jaccard (Content)':    {'vals': modal_data['NER Jaccard\n(Content)'],   'color': '#E53935', 'ls': '-',  'marker': 's'},
    'BERTScore F1 (Semantics)': {'vals': modal_data['BERTScore F1\n(Semantics)'],'color': '#43A047', 'ls': '--', 'marker': '^'},
    'SBERT Cosine (Semantics)': {'vals': modal_data['SBERT Cosine\n(Semantics)'],'color': '#7CB342', 'ls': '--', 'marker': 'v'},
    'BLEU-2 (Lexical)':         {'vals': modal_data['BLEU-2\n(Lexical)'],        'color': '#9E9E9E', 'ls': ':',  'marker': 'D'},
}

for label, cfg in plot_series.items():
    ax.plot(STAGES, cfg['vals'], cfg['marker'] + cfg['ls'],
            color=cfg['color'], label=label, linewidth=2.5, markersize=8)

ax.set_xlabel('Paraphrase Stage', fontsize=12)
ax.set_ylabel('Similarity to T0', fontsize=12)
ax.set_title('Multi-Modal Decay: Structure vs. Content vs. Semantics vs. Lexicon',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate the key insight
ax.annotate('Lexical skin replaced first',
            xy=(3, modal_data['BLEU-2\n(Lexical)'][3]), xytext=(2.3, 0.15),
            fontsize=9, fontstyle='italic', color='#616161',
            arrowprops=dict(arrowstyle='->', color='#9E9E9E', lw=1.2))

plt.tight_layout()
plt.savefig(FIGURES / "fig5_multimodal_audit.png", dpi=150, bbox_inches='tight')
plt.show()

---
## Summary of Findings

In [ ]:
print("=" * 70)
print("NER ANALYSIS SUMMARY — Content Erasure")
print("=" * 70)

print("\n1. ENTITY RETENTION DECAY")
for stage in ['T1', 'T2', 'T3']:
    j = combined[f'jaccard_{stage}'].mean()
    r = combined[f'recall_{stage}'].mean()
    p = combined[f'precision_{stage}'].mean()
    print(f"   {stage}: Jaccard={j:.3f}, Recall={r:.3f}, Precision={p:.3f}")

print("\n2. WORST ENTITY ERASERS (by Recall at T3)")
for family in family_colors:
    fam_df = combined[combined['family'] == family]
    r = fam_df['recall_T3'].mean()
    print(f"   {family.capitalize():10s}: Recall={r:.3f}")

print("\n3. WORST HALLUCINATORS (by Precision at T3)")
for family in family_colors:
    fam_df = combined[combined['family'] == family]
    p = fam_df['precision_T3'].mean()
    print(f"   {family.capitalize():10s}: Precision={p:.3f}")

print("\n4. MULTI-MODAL AUDIT")
print(f"   POS Cosine (structure) at T3: {modal_data['POS Cosine' + chr(10) + '(Structure)'][3]:.3f}")
print(f"   NER Jaccard (content)  at T3: {modal_data['NER Jaccard' + chr(10) + '(Content)'][3]:.3f}")
print(f"   BERTScore  (semantics) at T3: 0.810")
print(f"   BLEU-2     (lexical)   at T3: 0.280")
print("\n   Decay rate: Lexical > Content > Semantics > Structure")

print("\n" + "=" * 70)